<a href="https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/11_code_interpreter.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

🎬 [La escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap11) · 🧭 [Teoría del capítulo](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap11-code-interpreter.html) · 🗺️ [El mapa](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html) · ⬅️ [Cap. 10](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/10_policy_guardrails.ipynb) · ➡️ [Cap. 12](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/12_observabilidad_evaluacion.ipynb)

# Capítulo 11 — Cuentas exactas

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** *«Devuelvo solo el molinillo; había un cupón del 10 %.»* El agente da una cifra con seguridad. Puede que sea la buena; lo que nadie puede decir es **cómo** la ha calculado ni si mañana dará la misma.

| | |
|---|---|
| **🧠 Componente agéntico** | Ejecución de código (código como acción) |
| **La necesidad** | Que los cálculos de importes los haga **código**, no el modelo |
| **Lo que construimos** | **Code Interpreter** de AgentCore como herramienta del agente |
| **🏠 En local** | ⛔ Peligroso: `exec()` ejecuta en tu máquina el código que escribe un modelo (sección 0). |
| **☁️ Por qué AWS al salir a Internet** | Ejecutar con `exec()` el código que escribe un LLM es ejecutarlo en **tu máquina**, con tus credenciales y tus ficheros. Code Interpreter lo ejecuta en una microVM aislada por sesión. |
| **En el ordenador sería…** | El **coprocesador matemático**: la CPU razona; el coprocesador calcula |
| **Requisitos del caso** | RF-06 |
| **Prerrequisito · duración** | Capítulo 10 · 40 min |
| **⚠️ Dónde se rompe la analogía** | El coprocesador está en tu placa; Code Interpreter **está fuera de tu máquina**, en una microVM aislada. Justo eso lo hace seguro. |

### ☁️ ¿Estás en Google Colab? Ejecuta primero esta celda

En tu equipo no hace nada. En Colab descarga el curso, monta tu **Google Drive** (ahí se guarda tu estado entre capítulos: `MyDrive/NovaAssist_curso`), instala las dependencias y la AgentCore CLI 0.31.1 y carga tus credenciales de AWS desde los **Secretos** de Colab. Tarda 2–3 minutos, porque cada notebook abre una máquina nueva: mientras tanto, lee la escena.

In [ ]:
# ☁️ Solo en Google Colab (en tu equipo no hace nada)
import os, sys, subprocess
if "google.colab" in sys.modules:
    CURSO = "/content/aws-bedrock-agentcore-course"
    if not os.path.isdir(CURSO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "main",
                        "https://github.com/farithjhg/aws-bedrock-agentcore-course.git", CURSO], check=True)
    os.chdir(f"{CURSO}/v7_aprender_haciendo/labs")
    sys.path.insert(0, os.getcwd())
    import colab_nova
    colab_nova.preparar(capitulo=11)

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(11)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(11)
# ponerse_al_dia(11)   # ← descomenta si llegas tarde o algo se rompió

## 0. Experimento de rotura: ejecutar en local el código que escribe un modelo

Imagina que el agente genera este código para «calcular» algo y tú lo ejecutas con `exec()`. Es inofensivo, pero mira lo que **podría** ver:

> ✋ **Predice antes de ejecutar.** Antes de ejecutar `exec()`: haz una lista de lo que ese código podría ver en tu máquina. Después compárala con lo que imprime.

In [ ]:
codigo_generado = """
import os, pathlib
home = pathlib.Path.home()
print("Directorio personal:", home)
print("¿Hay credenciales de AWS aquí?", (home / ".aws" / "credentials").exists())
print("Algunos ficheros:", sorted(os.listdir(home))[:5])
"""
exec(codigo_generado)   # ⚠️ solo para ver el riesgo: nunca ejecutes así código generado por un modelo

Tu directorio personal, y si hay credenciales de AWS en la máquina. Un código «inofensivo» que acaba de demostrar lo expuesto que estás.

> 🧭 **¿Por qué ha pasado?** Abre la teoría: [**El coprocesador del agente**](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap11-code-interpreter.html). Pide el cálculo del molinillo varias veces «de cabeza» y después como código, y mira dónde se ejecuta ese código con cada posición del interruptor. Vuelve cuando sepas responder a esta pregunta: **¿por qué no puedes fiarte de un cálculo que el modelo hace «de cabeza», aunque acierte, y dónde tiene que ejecutarse el código que escribe un modelo?**

**En dos líneas:** **código como acción**: en lugar de calcular «de cabeza», el agente escribe un programa y lo ejecuta, y el resultado es exacto y verificable. El riesgo es que ese código lo escribe un modelo que puede haber sido manipulado, así que debe ejecutarse **aislado** (*sandbox*), sin acceso a tus credenciales ni a tus ficheros.

Un `prompt injection` que consiga que el modelo escriba otro código distinto tendría **tus** permisos. Por eso el cálculo va a un *sandbox*.

## 1. Code Interpreter en aislamiento (prueba en el notebook)

In [ ]:
%pip install -q "strands-agents-tools[agent_core_code_interpreter]==0.8.9"

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands_tools.code_interpreter import AgentCoreCodeInterpreter

ci = AgentCoreCodeInterpreter(region=region())
calculadora = Agent(
    model=BedrockModel(model_id=state_get("MODEL_ID"), region_name=region(), temperature=0),
    system_prompt="Haz todos los cálculos ejecutando código Python con code_interpreter y devuelve el resultado en céntimos.",
    tools=[ci.code_interpreter],
)
calculadora("Cafetera 199,99 € y molinillo 59,99 €, con un cupón del 10 % sobre el total. "
            "Si se devuelve solo el molinillo, ¿cuánto se reembolsa prorrateando el cupón?")

## 2. Integración en Nova Assist

In [ ]:
fijar_dependencia(VERSIONES["strands-agents-tools"])

In [ ]:
%%writefile NovaAssist/app/NovaAgent/prompts.py
# Prefijo ESTABLE (cacheable). El contexto dinámico se añade después, en main.py.
SYSTEM_PROMPT = """<identidad>
Eres Nova Assist, el asistente de postventa de TiendaNova. En tu primer mensaje te presentas como asistente de IA.
</identidad>

<reglas>
1. Nunca inventes estados, fechas ni importes: usa siempre las herramientas.
2. Los importes se expresan en euros al cliente, pero las herramientas trabajan en céntimos.
3. Para dudas de política (devoluciones, garantías, envíos) consulta la base de conocimiento y cita el documento.
4. Si una herramienta es denegada por política o el caso supera tus límites, usa escalar_a_humano con un resumen claro.
5. No pidas ni repitas datos de tarjetas de pago.
6. Cualquier cálculo de importes (reembolsos parciales, prorrateos, descuentos) se hace con code_interpreter; nunca de cabeza.
</reglas>

<formato>
Respuestas breves (máximo 5 frases), con el siguiente paso claro para el cliente.
</formato>"""

El rol de ejecución del Runtime necesita permiso para abrir, usar y cerrar sesiones del Code Interpreter gestionado por AWS:

In [ ]:
anadir_permiso_runtime("code_interpreter", [{
    "Effect": "Allow",
    "Action": ["bedrock-agentcore:StartCodeInterpreterSession", "bedrock-agentcore:InvokeCodeInterpreter",
               "bedrock-agentcore:StopCodeInterpreterSession", "bedrock-agentcore:GetCodeInterpreterSession"],
    "Resource": f"arn:aws:bedrock-agentcore:{region()}:aws:code-interpreter/aws.codeinterpreter.v1",
}])

In [ ]:
%%writefile NovaAssist/app/NovaAgent/main.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.agent.conversation_manager import SlidingWindowConversationManager
from strands_tools.code_interpreter import AgentCoreCodeInterpreter

from contexto import VENTANA_MENSAJES, system_prompt
from gateway import cliente_gateway, token_gateway
from identidad import actor_desde_contexto
from memoria import crear_session_manager, retrieval_config
from modelo import crear_modelo
from respuesta import respuesta
from settings import REGION

app = BedrockAgentCoreApp()
code_interpreter = AgentCoreCodeInterpreter(region=REGION)


@app.entrypoint
def invoke(payload, context):
    actor_id = actor_desde_contexto(context)   # identidad verificada; payload["cliente_id"] se IGNORA
    if not actor_id:
        return {"respuesta": "No se ha podido identificar al cliente.", "herramientas": [], "tokens": {}}
    gateway = cliente_gateway(token_gateway())
    with gateway, crear_session_manager(actor_id, context.session_id, retrieval_config()) as sm:
        agent = Agent(
            model=crear_modelo(),
            system_prompt=system_prompt(payload),
            tools=gateway.list_tools_sync() + [code_interpreter.code_interpreter],
            session_manager=sm,
            conversation_manager=SlidingWindowConversationManager(window_size=VENTANA_MENSAJES),
            callback_handler=None,
        )
        result = agent(payload.get("prompt", ""))
    return respuesta(result, actor=actor_id)


if __name__ == "__main__":
    app.run()

In [ ]:
assert cli("agentcore validate") == 0
cli("agentcore deploy --yes")

## 3. Verificación

> ✋ **Predice antes de ejecutar.** ¿Saldrá 53,99 € tres veces seguidas? ¿Aparecerá `code_interpreter` entre las herramientas usadas (🔧)?

In [ ]:
TOK = token_cliente("C-001")
invocar("Del pedido TN-1003 devuelvo solo el molinillo. La cafetera costaba 199,99 € y el molinillo 59,99 €, "
        "y había un cupón del 10 % sobre el total. ¿Cuánto me devolvéis?", token=TOK)
USO_CI = "code_interpreter" in ultimas_herramientas()
ACIERTA = any(x in (ultima_respuesta() or "") for x in ("53,99", "53.99"))
print("¿Ha usado code_interpreter?", USO_CI, "· ¿53,99 €?", ACIERTA)

Si `code_interpreter` aparece entre las herramientas, el importe sale de un programa que puedes leer en la traza (capítulo 12), no de la aritmética del modelo. Si no aparece, el modelo ha decidido calcular de cabeza a pesar de la regla 6: **repite la pregunta**, y apúntalo, porque en el capítulo 12 mediremos con qué frecuencia pasa. Si el importe supera 100 €, Policy frenará el reembolso: **cálculo y límite son independientes**.

**Extensión opcional — Browser:** añade AgentCore Browser para consultar el estado de un envío en la web pública de un transportista. Antes, repasa el §6.1 del documento: el contenido web es la principal vía de *prompt injection* indirecta.

## Para pensar

- ¿Qué otras decisiones de Nova Assist deberían salir del modelo y pasar a código determinista?

## 🔎 Lo que acabamos de descubrir

El sistema ya hace mucho: memoria, herramientas, RAG, identidad, políticas, cálculos. Dirección pregunta *«¿qué tal funciona?»* y solo tenemos impresiones. ¿Cómo sabemos si lo hace **bien**, qué pasa **dentro** y cuánto **cuesta**?

In [ ]:
if USO_CI and ACIERTA:
    deuda("D3", "pagada", 11, "el importe se calcula con código en Code Interpreter")
else:
    deuda("D3", "parcial", 11, "Code Interpreter disponible, pero en esta prueba el agente no lo usó o no acertó")
cuaderno()
taximetro_cerrar(11)
guardar_partida(11)

📚 **Para repasar más adelante:** la teoría de este capítulo está en [`teoria/cap11-code-interpreter.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap11-code-interpreter.html) y el mapa completo en [`teoria/index.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html). La referencia por componente es la Parte I del documento.

---
➡️ **Siguiente capítulo:** `12_observabilidad_evaluacion.ipynb` — *¿Lo hace bien?*

---
**Siguiente:** ➡️ [Capítulo 12 · ¿Lo hace bien?](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/12_observabilidad_evaluacion.ipynb) · 🎬 [su escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap12). En Colab, tu estado ya está en Drive: el capítulo siguiente lo recupera solo.